# 08 Task 4 results

Compares Telugu replies before the fix (notebook 02, hand-labeled) with replies under plain steering (notebook 07, labeled the same way) on the same prompts: plain, story and villain, plus the harmless twins for over-refusal.

CPU only. Set `BEFORE` to the labeled CSV from notebook 02 and `AFTER` to the labeled `responses_after_fix.csv`. Harmless replies in the after file are labeled `answered` or `refused`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact

BEFORE = ""
AFTER = ""
ATTACKS = ["plain", "story", "villain"]

def load(path):
    d = pd.read_csv(path).fillna("")
    d["label"] = d["label"].astype(str).str.strip()
    d["refusal"] = d["refusal"].astype(str).str.upper() == "TRUE"
    d.loc[(d["type"] == "harmful") & (d["label"] == "") & d["refusal"], "label"] = "refused"
    return d

before = load(BEFORE)
before = before[(before["lang"] == "telugu") &
                (((before["type"] == "harmful") & before["attack"].isin(ATTACKS)) |
                 ((before["type"] == "benign") & (before["attack"] == "plain")))].assign(method="none")
after = load(AFTER)
after = after[after["method"] == "plain_steer"]
both = pd.concat([before, after], ignore_index=True)

harm = both[both["type"] == "harmful"]
missing = harm[harm["label"] == ""]
print("Unlabeled harmful rows:", len(missing), missing.groupby("method").size().to_dict())
harm = harm[harm["label"] != ""].assign(hc=lambda x: x["label"] == "harmful_compliance")

table = harm.pivot_table(index="attack", columns="method", values="hc", aggfunc="mean").reindex(ATTACKS)
table.loc["all"] = harm.groupby("method")["hc"].mean()
print("\nTelugu harmful compliance, before vs after plain steering:")
print(table[["none", "plain_steer"]].round(2))

counts = harm.groupby("method")["hc"].agg(["sum", "count"])
_, p = fisher_exact([[counts.loc["none", "sum"], counts.loc["none", "count"] - counts.loc["none", "sum"]],
                     [counts.loc["plain_steer", "sum"], counts.loc["plain_steer", "count"] - counts.loc["plain_steer", "sum"]]])
print(f"Fisher exact test, all attacks: p = {p:.5f}")

print("\nLabel mix:")
print(pd.crosstab(harm["method"], harm["label"], normalize="index").round(2))

ben = both[both["type"] == "benign"].copy()
ben["over_refused"] = ben.apply(lambda r: r["label"] == "refused" if r["label"] in ("refused", "answered") else r["refusal"], axis=1)
print("\nOver-refusal on Telugu harmless prompts:")
print(ben.groupby("method")["over_refused"].agg(rate="mean", n="count").round(2))
print("(hand labels where present, keyword flag otherwise)")

ax = table[["none", "plain_steer"]].rename(columns={"none": "before", "plain_steer": "after (plain steering)"}).plot(
    kind="bar", figsize=(7, 4), rot=0)
ax.set_ylabel("Harmful compliance rate"); ax.set_ylim(0, 1)
ax.set_title("Tiny Aya Global, Telugu: before vs after steering")
plt.tight_layout(); plt.savefig("chart_task4_before_after.png", dpi=150); plt.show()
both.to_csv("task4_before_after.csv", index=False, encoding="utf-8-sig")